In [1]:
"""
05 - Visualisations for the report & presentation.

Generates:
- outputs/tsne_mobai_subgraph.png : 2D t-SNE with MoBai nodes highlighted
- outputs/variant_predictions_bar.png : bar chart of Variant A/B/C predictions
- outputs/sim_vs_sentiment_scatter.png : training data scatter (sim vs sentiment)
- outputs/ai_stack_diagram.png : 4-layer AI architecture diagram
"""
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import linregress

In [2]:
ROOT = Path("__file__" in globals() and __file__ or "notebooks/dummy.py").resolve().parents[1] if "__file__" in globals() else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"

In [3]:
with open(DATA / "flavorgraph_embeddings.pickle", "rb") as f:
    emb = pickle.load(f)
with open(DATA / "trained_models.pkl", "rb") as f:
    trained = pickle.load(f)
with open(DATA / "training_data.pkl", "rb") as f:
    td = pickle.load(f)

/tmp/ipykernel_136455/3354010768.py:2: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  emb = pickle.load(f)


In [4]:
nodes = pd.read_csv(DATA / "nodes_191120.csv")
nodes["node_id_str"] = nodes["node_id"].astype(str)
name_to_id = dict(zip(nodes["name"], nodes["node_id_str"]))
id_to_name = dict(zip(nodes["node_id_str"], nodes["name"]))

In [5]:
X_train, y_train, names_train = td["X"], td["y"], td["names"]
anchor = trained["top_anchor_centroid"]

In [6]:
# Try to load category info
try:
    cat_df = pd.read_csv(DATA / "dict_ingr2cate.csv")
    cat_map = dict(zip(cat_df["ingredient"], cat_df["category"]))
except FileNotFoundError:
    cat_map = {}

In [7]:
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titleweight": "bold",
})

In [8]:
# ========================================================
# (1) Scatter: training sim_to_anchor vs sentiment + Variant A/B overlay
# ========================================================
X_norm = X_train / np.linalg.norm(X_train, axis=1, keepdims=True)
sim_train = cosine_similarity(X_norm, anchor.reshape(1, -1)).flatten()
reg = linregress(sim_train, y_train)

In [9]:
# Variant predictions
def vec(name):
    nid = name_to_id.get(name)
    if nid is None or nid not in emb:
        return None
    v = np.asarray(emb[nid], dtype=float)
    return v / np.linalg.norm(v)

In [10]:
def combine(*names):
    vecs = [vec(n) for n in names if vec(n) is not None]
    v = np.mean(vecs, axis=0)
    return v / np.linalg.norm(v)

In [11]:
variant_pts = {
    "Variant A\n(Mango × Jasmine)": combine("fresh_mango", "jasmine_tea"),
    "Variant B\n(Coconut × Milk Tea)": combine("coconut", "black_tea", "milk"),
}

In [12]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(sim_train, y_train, s=90, c=y_train, cmap="RdYlGn", edgecolor="black", zorder=3)
for i, n in enumerate(names_train):
    ax.annotate(n, (sim_train[i], y_train[i]), xytext=(7, 4), textcoords="offset points", fontsize=9)
xs = np.linspace(sim_train.min() - 0.05, max(sim_train.max(), 0.85), 100)
ax.plot(xs, reg.slope * xs + reg.intercept, "--", c="steelblue", alpha=0.7,
        label=f"Linear fit (R²={reg.rvalue**2:.2f}, Pearson r={reg.rvalue:.2f})")
# Overlay variant predictions
for label, v in variant_pts.items():
    sim = float(cosine_similarity(v.reshape(1, -1), anchor.reshape(1, -1))[0, 0])
    pred = reg.slope * sim + reg.intercept
    ax.scatter([sim], [pred], marker="*", s=400, c="gold", edgecolor="black", zorder=4)
    ax.annotate(label, (sim, pred), xytext=(8, -20), textcoords="offset points",
                fontsize=10, fontweight="bold", color="darkred")
ax.set_xlabel("Cosine similarity to high-sentiment anchor (FlavorGraph embedding)")
ax.set_ylabel("Net consumer sentiment (%, from NLP 5,021 tweets)")
ax.set_title("Liking-Score Calibration Curve\nFlavorGraph similarity predicts consumer sentiment (Pearson r=0.60)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "sim_vs_sentiment_scatter.png", bbox_inches="tight")
plt.close()
print(f"Saved: sim_vs_sentiment_scatter.png")

Saved: sim_vs_sentiment_scatter.png


In [13]:
# ========================================================
# (2) Variant predictions bar chart
# ========================================================
pred_df = pd.read_csv(OUT / "pairing_predictions.csv")
# Pick a curated subset for the chart
pick_labels = [
    "Variant B · Full product (yogurt + coconut + black_tea + milk)",
    "Variant B · Coconut × Milk Tea",
    "Variant A · Mango × Jasmine (food node)",
    "Variant A · Full product (yogurt + mango + jasmine_tea)",
    "Baseline · Mango alone",
    "Baseline · Vanilla alone (top performer in NLP)",
    "Baseline · Strawberry alone (lowest performer in NLP)",
]
sub = pred_df[pred_df["label"].isin(pick_labels)].copy()
sub["short_label"] = sub["label"].str.replace("Variant A · ", "Var A · ").str.replace("Variant B · ", "Var B · ").str.replace("Baseline · ", "BL · ")
sub = sub.sort_values("predicted_sentiment", ascending=True).reset_index(drop=True)
colors = sub["predicted_sentiment"].apply(lambda v: "#2ca02c" if v >= 45 else ("#1f77b4" if v >= 38 else ("#ff7f0e" if v >= 30 else "#d62728")))

In [14]:
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(sub["short_label"], sub["predicted_sentiment"], color=colors, edgecolor="black")
for bar, val, sim in zip(bars, sub["predicted_sentiment"], sub["sim_to_anchor"]):
    ax.text(val + 0.4, bar.get_y() + bar.get_height() / 2,
            f" {val:.1f}%  (sim={sim:.2f})", va="center", fontsize=9)
ax.axvline(x=y_train.mean(), color="gray", linestyle="--", alpha=0.6, label=f"Training mean ({y_train.mean():.0f}%)")
ax.set_xlim(0, 60)
ax.set_xlabel("Predicted net sentiment score (%)")
ax.set_title("AI-Predicted Consumer Liking — MoBai Variants vs Baselines")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "variant_predictions_bar.png", bbox_inches="tight")
plt.close()
print(f"Saved: variant_predictions_bar.png")

Saved: variant_predictions_bar.png


In [15]:
# ========================================================
# (3) Variant C top 10 chart
# ========================================================
vc = pd.read_csv(OUT / "variant_c_curated_top20.csv").head(10).iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(vc["name"], vc["predicted_sentiment"], color="#6baed6", edgecolor="black")
for bar, val, cat in zip(bars, vc["predicted_sentiment"], vc["category"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f" {val:.1f}%  · {cat}", va="center", fontsize=9)
ax.set_xlabel("Predicted net sentiment score (%) when paired with yogurt + black tea base")
ax.set_title("Variant C Discovery — Top 10 Beverage-Friendly Candidates\n(screened from 8,279 FlavorGraph nodes, filtered by category)")
ax.set_xlim(30, 50)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "variant_c_top10.png", bbox_inches="tight")
plt.close()
print(f"Saved: variant_c_top10.png")

Saved: variant_c_top10.png


In [16]:
# ========================================================
# (4) t-SNE projection
# ========================================================
# Project a focused subset (training flavors + MoBai-related + Variant C top + ~500 random)
focus_names = list(names_train) + [
    "jasmine_tea", "Benzyl_Acetate", "Linalool",
    "yogurt", "Greek_yogurt" if "Greek_yogurt" in name_to_id else "plain_yogurt",
    "white_peach", "lychee", "passion_fruit",
]
# Add top variant C candidates
focus_names += list(pd.read_csv(OUT / "variant_c_curated_top20.csv").head(10)["name"])
focus_ids = set()
for n in focus_names:
    nid = name_to_id.get(n)
    if nid and nid in emb:
        focus_ids.add(nid)

In [17]:
# Plus random sample for context
rng = np.random.default_rng(42)
all_ids = [i for i in emb.keys() if i not in focus_ids]
sample_ids = rng.choice(all_ids, size=400, replace=False).tolist()

In [18]:
plot_ids = list(focus_ids) + sample_ids
vectors = np.stack([emb[i] for i in plot_ids])
labels = [id_to_name.get(i, i) for i in plot_ids]

In [19]:
print(f"Running t-SNE on {len(plot_ids)} nodes...")
ts = TSNE(n_components=2, random_state=42, perplexity=30, init="pca")
coords = ts.fit_transform(vectors)

Running t-SNE on 429 nodes...


In [20]:
fig, ax = plt.subplots(figsize=(13, 9))
# Background: random sample
ax.scatter(coords[len(focus_ids):, 0], coords[len(focus_ids):, 1],
           c="lightgray", s=15, alpha=0.5, label=f"Other nodes (n={len(sample_ids)})")

In [21]:
# Foreground: focused nodes
training_set = set(names_train)
variant_c_set = set(pd.read_csv(OUT / "variant_c_curated_top20.csv").head(10)["name"])
mobai_set = {"jasmine_tea", "Benzyl_Acetate", "Linalool", "white_peach", "lychee", "passion_fruit"}

In [22]:
for i, name in enumerate(labels[:len(focus_ids)]):
    x, yc = coords[i]
    if name in training_set:
        ax.scatter(x, yc, c="orange", s=120, edgecolor="black", zorder=3, label="_nlp_train")
        ax.annotate(name, (x, yc), xytext=(5, 5), textcoords="offset points", fontsize=8, fontweight="bold")
    elif name in mobai_set:
        ax.scatter(x, yc, c="red", s=140, marker="*", edgecolor="black", zorder=4, label="_mobai")
        ax.annotate(name, (x, yc), xytext=(5, 5), textcoords="offset points", fontsize=8, color="darkred")
    elif name in variant_c_set:
        ax.scatter(x, yc, c="green", s=100, marker="D", edgecolor="black", zorder=3, label="_var_c")
        ax.annotate(name, (x, yc), xytext=(5, 5), textcoords="offset points", fontsize=8, color="darkgreen")

In [23]:
# Manual legend
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray", markersize=8, label="Other nodes (random sample)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="orange", markeredgecolor="black", markersize=10, label="NLP training flavors (13)"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="red", markeredgecolor="black", markersize=14, label="MoBai ingredients (Variant A/B)"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor="green", markeredgecolor="black", markersize=9, label="Variant C top-10 candidates"),
]
ax.legend(handles=legend_elems, loc="upper right")
ax.set_title("FlavorGraph 2D t-SNE projection — MoBai-relevant nodes highlighted\n(300D embeddings → 2D via t-SNE; semantic clusters visible)")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(OUT / "tsne_mobai_subgraph.png", bbox_inches="tight")
plt.close()
print(f"Saved: tsne_mobai_subgraph.png")

Saved: tsne_mobai_subgraph.png


In [24]:
# ========================================================
# (5) AI Stack Diagram (4-layer architecture)
# ========================================================
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis("off")

(np.float64(0.0), np.float64(12.0), np.float64(0.0), np.float64(8.0))

In [25]:
layers = [
    {"y": 6.2, "color": "#fde68a", "title": "LAYER 1 · Consumer Voice", "subtitle": "VADER + TextBlob sentiment · TF-IDF · K-Means segmentation",
     "input": "5,021 tweets", "output": "→ flavor sentiment scores per ingredient (13 flavors)"},
    {"y": 4.5, "color": "#bae6fd", "title": "LAYER 2 · Molecular AI (FlavorGraph)", "subtitle": "metapath2vec + Chemical Structure Prediction (Park et al., 2021)",
     "input": "8,297 food + compound nodes, 147K edges", "output": "→ 300-dim embedding per ingredient"},
    {"y": 2.8, "color": "#bbf7d0", "title": "LAYER 3 · Liking-Score Predictor (our contribution)", "subtitle": "Cosine similarity to high-sentiment anchor · Linear calibration",
     "input": "{Layer 1 sentiments} + {Layer 2 embeddings}", "output": "→ predicted liking score for any flavor/combination"},
    {"y": 1.1, "color": "#fecaca", "title": "LAYER 4 · Product Science", "subtitle": "5-mechanism off-note masking · Sensory science · Peer-reviewed lit (Best 2025, Tian 2020, JAFC 2024)",
     "input": "Variant A/B from Layer 3", "output": "→ MoBai 茉白 product formulation"},
]

In [26]:
for L in layers:
    box = FancyBboxPatch((0.4, L["y"] - 0.55), 11.2, 1.1,
                         boxstyle="round,pad=0.05", linewidth=1.5,
                         facecolor=L["color"], edgecolor="black")
    ax.add_patch(box)
    ax.text(0.7, L["y"] + 0.3, L["title"], fontsize=12, fontweight="bold")
    ax.text(0.7, L["y"] + 0.0, L["subtitle"], fontsize=9, style="italic", color="#374151")
    ax.text(0.7, L["y"] - 0.3, f"IN: {L['input']}  {L['output']}", fontsize=9, color="#111827")

In [27]:
# Arrows
for i in range(len(layers) - 1):
    y_from = layers[i]["y"] - 0.6
    y_to = layers[i + 1]["y"] + 0.6
    ax.annotate("", xy=(6, y_to), xytext=(6, y_from),
                arrowprops=dict(arrowstyle="->", lw=2, color="#374151"))

In [28]:
ax.text(6, 7.5, "MoBai AI Pipeline · 4-Layer Architecture", fontsize=15,
        fontweight="bold", ha="center")
ax.text(6, 7.1, "From consumer language → molecular embedding → liking prediction → product formulation", fontsize=10, ha="center", style="italic", color="#4b5563")

Text(6, 7.1, 'From consumer language → molecular embedding → liking prediction → product formulation')

In [29]:
plt.savefig(OUT / "ai_stack_diagram.png", bbox_inches="tight")
plt.close()
print(f"Saved: ai_stack_diagram.png")

Saved: ai_stack_diagram.png


/tmp/ipykernel_136455/3360771809.py:1: UserWarning: Glyph 33545 (\N{CJK UNIFIED IDEOGRAPH-8309}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT / "ai_stack_diagram.png", bbox_inches="tight")
/tmp/ipykernel_136455/3360771809.py:1: UserWarning: Glyph 30333 (\N{CJK UNIFIED IDEOGRAPH-767D}) missing from font(s) DejaVu Sans.
  plt.savefig(OUT / "ai_stack_diagram.png", bbox_inches="tight")


In [30]:
print("\nAll visualisations done.")


All visualisations done.
